# Prompt Template: Proposer (Raw (No Personality))

Prompt construction notebook for the proposer role using raw (no personality) prompts.

**Related text file:** `_proposer_raw.txt`

In [7]:
import json
import logging
import sys
import time
import subprocess
import threading
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, Any

import pandas as pd
import re
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor, as_completed

import langchain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM

"""
Ultimatum-game proposer simulation WITHOUT personality (no d-traits / D factor).
- No personality CSV is loaded.
- Agents do not have d or d_description fields.
- The LLM is invoked with ONLY situation variables (situation, keep, send, fair).
- If your prompt template still contains {d_value} / {d_description}, they are
  passed as empty strings to avoid template errors, but no trait information is
  actually provided to the model.
"""

MODEL_NAME = "phi4"  # change this to whichever Ollama model you want
TEMPERATURE = 0.8
N_AGENTS = 1000  # how many independent runs/agents to simulate (formerly REPEATS semantics)
MAX_WORKERS = 1  # concurrency level
INVOKE_TIMEOUT = 240  # per-call timeout seconds
MAX_RETRIES = 2  # number of additional retry attempts
BASE_BACKOFF = 1.0  # backoff base in seconds
RESTART_THROTTLE_SEC = 30  # throttle restarts

# File paths
PROMPT_PATH = "_proposer_raw.txt"  # make sure this version does NOT mention traits
QUESTION_CSV = "_proposer_ultimatum_questions.csv"
OUTPUT_CSV = f"_data_proposer_{MODEL_NAME.replace(':', '_').replace('-', '_')}_nopersonality_temp{TEMPERATURE}_{N_AGENTS}.csv"
MALFORMED_RESPONSES_LOG = "malformed_responses_samples.jsonl"

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s",
                    handlers=[logging.StreamHandler(sys.stdout)])
base_logger = logging.getLogger("sim")
langchain.verbose = True


class ContextAdapter(logging.LoggerAdapter):
    def process(self, msg, kwargs):
        extra = self.extra.copy()
        parts = [f"{k}={v}" for k, v in extra.items() if v is not None]
        prefix = "[" + " ".join(parts) + "] " if parts else ""
        return prefix + msg, kwargs


def make_logger(agent_id=None, question_nr=None, retry=None):
    return ContextAdapter(base_logger, {"agent_id": agent_id, "question_nr": question_nr, "retry": retry})


ollama_restarting_event = threading.Event()
restart_lock = threading.Lock()
_last_restart_time = 0.0


def is_ollama_running() -> bool:
    try:
        output = subprocess.check_output(["tasklist", "/FI", "IMAGENAME eq ollama.exe"], stderr=subprocess.STDOUT)
        return b"ollama.exe" in output
    except Exception as e:
        base_logger.error(f"Error checking Ollama process: {e}")
        return False


def restart_ollama():
    global _last_restart_time
    with restart_lock:
        now = time.time()
        if now - _last_restart_time < RESTART_THROTTLE_SEC:
            base_logger.info("Restart throttled; skipping.")
            return
        _last_restart_time = now

        base_logger.info("Restarting Ollama...")
        try:
            subprocess.run(["taskkill", "/F", "/IM", "ollama.exe"], stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                           check=True)
        except Exception as e:
            base_logger.warning(f"Failed to kill Ollama cleanly: {e}")

        wait_time = 0
        while is_ollama_running() and wait_time < 20:
            base_logger.info("Waiting for Ollama to shutdown...")
            time.sleep(2)
            wait_time += 2
        if is_ollama_running():
            base_logger.error("Ollama did not terminate in time.")
        else:
            base_logger.info("Ollama terminated.")

        try:
            subprocess.Popen(["ollama", "start"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except Exception as e:
            base_logger.error(f"Error starting Ollama: {e}")

        wait_time = 0
        while not is_ollama_running() and wait_time < 20:
            base_logger.info("Waiting for Ollama to start...")
            time.sleep(2)
            wait_time += 2
        if not is_ollama_running():
            base_logger.error("Ollama did not start in time.")
        else:
            base_logger.info("Ollama restarted successfully.")


def wait_for_ollama():
    while ollama_restarting_event.is_set():
        time.sleep(0.5)


@dataclass
class Agent:
    ID: int
    Token: str


@dataclass
class Situation:
    question_nr: Any
    situation: str
    keep: Any
    send: Any
    fair: Any


def load_csv_validated(path: str, required_cols: list):
    """Load CSV trying several encodings to avoid Windows cp1252 issues."""
    encodings = ("utf-8", "utf-8-sig", "cp1252", "cp1250", "latin-1")
    last_err = None
    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc)
            missing = [c for c in required_cols if c not in df.columns]
            if missing:
                raise ValueError(f"CSV {path} missing required columns: {missing}")
            return df
        except UnicodeDecodeError as e:
            last_err = e
            continue
    if last_err:
        raise UnicodeDecodeError(
            last_err.encoding or "unknown",
            last_err.object if hasattr(last_err, "object") else b"",
            getattr(last_err, "start", 0),
            getattr(last_err, "end", 0),
            f"Failed to decode {path} with common encodings. Please re-save as UTF-8 without BOM."
        )
    raise ValueError(f"Failed to load {path} for unknown reasons.")


def load_prompt(file_path: str) -> str:
    """Read prompt text robustly across Windows encodings.
    Tries common encodings; last resort replaces undecodable bytes.
    """
    encodings = ("utf-8", "utf-8-sig", "cp1252", "cp1250", "latin-1")
    last_err = None
    for enc in encodings:
        try:
            return Path(file_path).read_text(encoding=enc)
        except UnicodeDecodeError as e:
            last_err = e
            continue
    # Fallback that never throws, but may replace unknown bytes
    try:
        return Path(file_path).read_text(encoding="latin-1", errors="replace")
    except Exception as e:  # truly unexpected
        raise e if last_err is None else last_err


def normalize_decision(dec: str) -> Optional[str]:
    if not dec:
        return None
    dec = dec.strip().upper()
    return dec if dec in ("A", "B") else None


def save_malformed_sample(agent_id, question_nr, raw_response, parsed_decision, parsed_justification):
    sample = {
        "timestamp": time.time(),
        "agent_id": agent_id,
        "question_nr": question_nr,
        "raw_response": raw_response,
        "parsed_decision": parsed_decision,
        "parsed_justification": parsed_justification
    }
    with open(MALFORMED_RESPONSES_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(sample) + "\n")


def parse_response(response: str) -> Tuple[Optional[str], Optional[str]]:
    try:
        stripped = response.strip()
        if stripped.startswith("{"):
            obj = json.loads(stripped)
            decision = normalize_decision(obj.get("decision", ""))
            justification = obj.get("justification", "").strip()
            if decision and justification:
                return decision, justification
        decision_match = re.search(r'Decision:\s*([ABab])', response)
        justification_match = re.search(r'Justification:\s*(.+)', response, re.DOTALL)
        if decision_match and justification_match:
            decision = normalize_decision(decision_match.group(1))
            justification = justification_match.group(1).strip()
            return decision, justification
    except Exception:
        pass
    return None, None


# LLM and prompt chain (no trait conditioning)
llm = OllamaLLM(model=MODEL_NAME, temperature=TEMPERATURE, num_predict=4096)
prompt_sender = load_prompt(PROMPT_PATH)
template_s = ChatPromptTemplate.from_template(prompt_sender)
proposer_chain = template_s | llm | StrOutputParser()


def build_payload_for_template(sit: Situation) -> Dict[str, Any]:
    """Construct payload strictly from situation fields.
    If the template (by mistake) still requires d_value or d_description, feed empty strings
    so no personality information leaks, while preventing KeyError.
    """
    payload = {
        "situation": sit.situation,
        "keep": sit.keep,
        "send": sit.send,
        "fair": sit.fair,
    }
    required = set(getattr(template_s, "input_variables", []))
    if "d_value" in required:
        payload["d_value"] = ""
    if "d_description" in required:
        payload["d_description"] = ""
    return payload


def invoke_with_retries(payload: Dict[str, Any], context: Dict[str, Any]) -> Tuple[str, Optional[str], Optional[str]]:
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 2):
        log = make_logger(agent_id=context.get("agent_id"), question_nr=context.get("question_nr"), retry=attempt)
        try:
            wait_for_ollama()
            with ThreadPoolExecutor(max_workers=1) as exec_single:
                future = exec_single.submit(lambda: proposer_chain.invoke(payload))
                raw = future.result(timeout=INVOKE_TIMEOUT)
            decision, justification = parse_response(raw)
            if decision and justification:
                return raw, decision, justification
            log.warning("Malformed output; saving sample.")
            save_malformed_sample(context.get("agent_id"), context.get("question_nr"), raw, decision, justification)
            if attempt > MAX_RETRIES:
                return raw, decision, justification
            backoff = BASE_BACKOFF * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            time.sleep(backoff)
        except concurrent.futures.TimeoutError:
            log.error("Timeout on invoke.")
            last_exc = "timeout"
            if attempt == 1:
                ollama_restarting_event.set()
                restart_ollama()
                ollama_restarting_event.clear()
            backoff = BASE_BACKOFF * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            time.sleep(backoff)
        except Exception as e:
            log.error(f"Unexpected error: {e}")
            last_exc = e
            if attempt == 1:
                ollama_restarting_event.set()
                restart_ollama()
                ollama_restarting_event.clear()
            backoff = BASE_BACKOFF * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            time.sleep(backoff)
    return f"Error after retries: {last_exc}", None, None


def process_sender(agent: Agent, situations: list) -> Dict[str, Any]:
    result: Dict[str, Any] = {
        "agent_id": agent.ID,
        "token": agent.Token,
    }
    for sit in situations:
        context = {"agent_id": agent.ID, "question_nr": sit.question_nr}
        log = make_logger(agent_id=agent.ID, question_nr=sit.question_nr, retry=None)
        payload = build_payload_for_template(sit)
        log.info(f"Invoking for question {sit.question_nr}")
        raw_resp, decision, justification = invoke_with_retries(payload, context)
        if decision is None or justification is None:
            log.warning(f"Final parse failure. decision={decision} justification={justification}")
        result[f"q{sit.question_nr}_full_response"] = raw_resp
        result[f"q{sit.question_nr}_decision"] = decision
        result[f"q{sit.question_nr}_text"] = justification
    return result


def simulate_responses_sender(agents_df: pd.DataFrame, situation_df: pd.DataFrame) -> pd.DataFrame:
    agents = []
    for _, row in agents_df.iterrows():
        agents.append(Agent(ID=row["ID"], Token=row["Token"]))
    situations = []
    for _, row in situation_df.iterrows():
        situations.append(Situation(question_nr=row.get("question_nr"),
                                    situation=row.get("situation"),
                                    keep=row.get("keep"),
                                    send=row.get("send"),
                                    fair=row.get("fair")))
    results = []
    total = len(agents)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_map = {executor.submit(process_sender, ag, situations): ag for ag in agents}
        for i, future in enumerate(as_completed(future_map), 1):
            agent = future_map[future]
            try:
                res = future.result()
                results.append(res)
            except Exception as exc:
                make_logger(agent_id=agent.ID).error(f"Agent crashed: {exc}")
            if i % max(1, total // 4) == 0 or i == total:
                base_logger.info(f"Progress {i}/{total} agents ({(i / total) * 100:.1f}%)")
    return pd.DataFrame(results)


if __name__ == "__main__":
    # Load questions only (no personality file)
    ultimatum_questionnaire = load_csv_validated(
        QUESTION_CSV,
        required_cols=['question_nr', 'situation', 'keep', 'send', 'fair']
    )

    # Build neutral agents (no d / personality). One row per independent run.
    agents_df = pd.DataFrame({
        'ID': range(1, N_AGENTS + 1),
        'Token': [f"R{i}" for i in range(1, N_AGENTS + 1)],
    })

    # Run the simulation
    simulation_results = simulate_responses_sender(agents_df, ultimatum_questionnaire)
    simulation_results.to_csv(OUTPUT_CSV, index=False)
    print(simulation_results.head())


2025-08-19 11:22:30,330 - [agent_id=1 question_nr=1] Invoking for question 1
2025-08-19 11:22:30,373 - HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"
2025-08-19 11:22:31,111 - [agent_id=2 question_nr=1] Invoking for question 1
2025-08-19 11:22:31,147 - HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"
2025-08-19 11:22:31,640 - [agent_id=3 question_nr=1] Invoking for question 1
2025-08-19 11:22:31,675 - HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"
2025-08-19 11:22:32,330 - [agent_id=4 question_nr=1] Invoking for question 1
2025-08-19 11:22:32,368 - HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"
2025-08-19 11:22:32,959 - [agent_id=5 question_nr=1] Invoking for question 1
2025-08-19 11:22:32,994 - HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"
2025-08-19 11:22:33,613 - [agent_id=6 question_nr=1] Invoking for question 1
2025-08-19 11:22:33,649 - HTTP Request: POS